In [1]:
from pathlib import Path
import sys
import workplace_pipeline as pipeline

project_dir = Path.cwd()

required_files = [
    "config/integrated/pipeline_v1.json",
    "data/evaluation/development_v1.json",
    "data/access_control/policy_v1.json",
]

missing = [
    name for name in required_files
    if not (project_dir / name).is_file()
]

if missing:
    raise RuntimeError(f"Missing project files: {missing}")

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Module loaded:", pipeline.__file__)
print("Integrated rollback prerequisites found:", len(required_files))

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Module loaded: C:\Users\Khaleel\workplace-ai-operations-lab\workplace_pipeline.py
Integrated rollback prerequisites found: 3


In [2]:
import json
from uuid import uuid4

rollback_dataset = json.loads(
    (project_dir / "data/evaluation/development_v1.json").read_text(
        encoding="utf-8"
    )
)

rollback_run_dir = (
    project_dir / "evidence" / "integrated_rollback" / uuid4().hex
)
rollback_run_dir.mkdir(parents=True, exist_ok=False)

active_config_path = rollback_run_dir / "active_configuration.json"
rollback_log_path = rollback_run_dir / "requests.jsonl"

known_good_bytes = (
    project_dir / "config/integrated/pipeline_v1.json"
).read_bytes()

active_config_path.write_bytes(known_good_bytes)
active_config_relative = active_config_path.relative_to(
    project_dir
).as_posix()


def run_rollback_cases():
    observations = []

    for case in rollback_dataset["cases"]:
        response = pipeline.retrieve_request(
            project_root=project_dir,
            config_relative_path=active_config_relative,
            question=case["question"],
            role="employee",
            log_path=rollback_log_path,
        )

        observations.append({
            "case_id": case["id"],
            "response": response,
        })

    return observations


def retrieval_signature(observations):
    # Exclude timestamps, request IDs, and elapsed times.
    return [
        {
            "case_id": item["case_id"],
            "config_version": item["response"]["config_version"],
            "status": item["response"]["status"],
            "error_code": item["response"]["error_code"],
            "result_count": item["response"]["result_count"],
            "results": item["response"]["results"],
        }
        for item in observations
    ]


baseline_observations = run_rollback_cases()
baseline_signature = retrieval_signature(baseline_observations)

assert len(baseline_observations) == 9
assert all(
    item["response"]["status"] in {"ok", "no_match"}
    and item["response"]["error_code"] is None
    for item in baseline_observations
)

for item in baseline_observations:
    print(
        f"{item['case_id']}: "
        f"{item['response']['status']} | "
        f"results={item['response']['result_count']}"
    )

print("\nKnown-good cases captured:", len(baseline_signature))
print("Run folder:", rollback_run_dir)

DEV-001: ok | results=2
DEV-002: ok | results=1
DEV-003: ok | results=2
DEV-004: no_match | results=0
DEV-005: no_match | results=0
DEV-006: ok | results=2
DEV-007: ok | results=2
DEV-008: ok | results=1
DEV-009: no_match | results=0

Known-good cases captured: 9
Run folder: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\integrated_rollback\2e02379d799d4bf2bbf82e4baafcdc03


In [3]:
import os
from datetime import datetime, timezone

rollback_timeline = []


def record_rollback_event(action, **details):
    rollback_timeline.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "action": action,
        **details,
    })


def replace_active_configuration(content):
    temporary_path = rollback_run_dir / f"pending_{uuid4().hex}.json"
    temporary_path.write_bytes(content)
    os.replace(temporary_path, active_config_path)


bad_configuration = json.loads(known_good_bytes.decode("utf-8"))
bad_configuration["version"] = "invalid-test"
bad_configuration["top_k"] = 0
bad_configuration_bytes = (
    json.dumps(bad_configuration, indent=2) + "\n"
).encode("utf-8")

record_rollback_event("baseline_captured", cases=len(baseline_signature))

try:
    # Deliberate validation bypass for this incident simulation.
    replace_active_configuration(bad_configuration_bytes)
    record_rollback_event("invalid_configuration_activated")

    failure_observations = run_rollback_cases()

    assert len(failure_observations) == 9
    assert all(
        item["response"]["status"] == "error"
        and item["response"]["error_code"] == "INVALID_CONFIGURATION"
        and item["response"]["result_count"] == 0
        and item["response"]["results"] == []
        for item in failure_observations
    )

    record_rollback_event(
        "failure_detected",
        rejected_cases=len(failure_observations),
        error_code="INVALID_CONFIGURATION",
    )

finally:
    replace_active_configuration(known_good_bytes)
    record_rollback_event("known_good_configuration_restored")

recovery_observations = run_rollback_cases()
recovery_signature = retrieval_signature(recovery_observations)

bytes_restored = active_config_path.read_bytes() == known_good_bytes
results_restored = recovery_signature == baseline_signature

assert bytes_restored
assert results_restored

record_rollback_event(
    "recovery_verified",
    bytes_restored=bytes_restored,
    per_case_results_restored=results_restored,
    cases=len(recovery_signature),
)

print("Invalid configuration rejected for:", len(failure_observations), "cases")
print("Configuration bytes restored:", bytes_restored)
print("All nine per-case results restored:", results_restored)
print("Timeline events:", len(rollback_timeline))

Invalid configuration rejected for: 9 cases
Configuration bytes restored: True
All nine per-case results restored: True
Timeline events: 5


In [4]:
import hashlib

all_observations = (
    baseline_observations
    + failure_observations
    + recovery_observations
)

rollback_events = [
    json.loads(line)
    for line in rollback_log_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(all_observations) == len(rollback_events) == 27
assert len({event["request_id"] for event in rollback_events}) == 27

for item in all_observations:
    response = item["response"]
    matching = [
        event for event in rollback_events
        if event["request_id"] == response["request_id"]
    ]

    assert len(matching) == 1
    assert matching[0] == {
        key: value for key, value in response.items()
        if key != "results"
    }


def sha256_bytes(content):
    return hashlib.sha256(content).hexdigest()


# Preserve the faulty candidate as evidence, separate from the active file.
candidate_path = rollback_run_dir / "invalid_candidate.json"

if candidate_path.exists():
    if candidate_path.read_bytes() != bad_configuration_bytes:
        raise RuntimeError("Existing candidate evidence differs.")
else:
    candidate_path.write_bytes(bad_configuration_bytes)

rollback_report = {
    "exercise_type": "Simulated integrated configuration incident",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "role": "employee; simulated input",
    "changed_setting": {"name": "top_k", "before": 2, "after": 0},
    "validation_bypassed": True,
    "timeline": rollback_timeline,
    "recovery": {
        "configuration_bytes_restored": bytes_restored,
        "per_case_results_restored": results_restored,
        "compared_cases": len(baseline_signature),
    },
    "fingerprints": {
        "known_good_configuration": sha256_bytes(known_good_bytes),
        "invalid_candidate": sha256_bytes(bad_configuration_bytes),
        "module": sha256_bytes(Path(pipeline.__file__).read_bytes()),
        "dataset": sha256_bytes(
            (project_dir / "data/evaluation/development_v1.json").read_bytes()
        ),
        "request_log": sha256_bytes(rollback_log_path.read_bytes()),
    },
    "verified_log_events": len(rollback_events),
    "baseline": baseline_observations,
    "failure": failure_observations,
    "recovery_observations": recovery_observations,
    "limitations": [
        "Development cases, not held-out evaluation.",
        "Recovery restores known baseline limitations.",
        "No live service, concurrency, or filesystem failure injection.",
        "No deployment validation gate implemented.",
        "Source documents and policy are not fingerprinted in this report.",
        "Timeline is persisted after completion.",
    ],
}

rollback_report_path = rollback_run_dir / "rollback_report.json"

with rollback_report_path.open("x", encoding="utf-8") as file:
    json.dump(rollback_report, file, indent=2, ensure_ascii=False)

print("Verified request/log pairs:", len(rollback_events))
print("Rollback report saved:", rollback_report_path)
print("Per-case recovery verified:", results_restored)

Verified request/log pairs: 27
Rollback report saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\integrated_rollback\2e02379d799d4bf2bbf82e4baafcdc03\rollback_report.json
Per-case recovery verified: True
